In [2]:


import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import random
from collections import defaultdict
from genomic_benchmarks.dataset_getters.pytorch_datasets import get_dataset
# Import the genomic benchmark dataset from Hugging Face
from genomic_benchmarks.dataset_getters.pytorch_datasets import DemoMouseEnhancers

# --------------------
# Data Preprocessing
# --------------------




    # Load the genomic benchmark dataset from Hugging Face
train_dset = get_dataset("human_enhancers_ensembl", split="train", version=0)
# test_dset  = get_dataset("human_enhancers_ensembl", split="test", version=0)

    # Extract sequences and labels from the datasets
train_seqs = [item[0] for item in train_dset]
train_labels = [item[1] for item in train_dset]
# test_seqs = [item[0] for item in test_dset]
# test_labels = [item[1] for item in test_dset]

    



In [3]:
pip install triton-windows

Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
# converting the lists to train and test csv
train_df = pd.DataFrame({
    'sequence':train_seqs,
    'target': train_labels
})

In [5]:
test_df = pd.read_csv(r"D:\genomic benchmark\human_enhancers_ensembl\dnabert-2\ienhancertest.csv")

In [6]:
train_df.head(5)

,sequence,target
0,AAGGCCAAGTGGCGTTTTGGCAGGCGGGCAGCAGGGTGGCACTGCT...,0
1,AGAAACAACCCACAGAGTAGGAGAAAGTATTTGCCAAATCATACAT...,0
2,CTCATCTCAGGGTCTCCTTTTAGAGGAATACAAGCTAAGGAACTCC...,0
3,AGTACAGTGGGACACAAGGTCATAGTGTTGGGTGTTAAGAACAAAG...,0
4,GGGATGGTGCTAAACCATTCGTGAGAGATCCGCCCCCATGATCCAA...,0


In [7]:
train_df.shape

(123872, 2)

In [8]:
test_df.head(5)

,Unnamed: 0,sequence,target,embedding
0,0,AATTTTCTCATTTTCTCATAAAGTTTAACAGTTGTTTATTTGAGTC...,0,[-5.25533855e-02 3.42247225e-02 4.01015058e-...
1,1,ACTGGTTATCTTTTAGGACTAGTTAATATAACCCATTCTCTAACCA...,0,[-4.27325815e-02 6.18737116e-02 2.17199940e-...
2,2,ATGCATATGTTCTTCAGTAAACAGAGCAGCCACTGGTACCACAGGA...,0,[-6.47496805e-02 8.43824297e-02 5.79394288e-...
3,3,CTGCTCTCCTCGCTCTATAAAAGTCAGAGTGCCTAAGCTGTTAATT...,0,[-7.63327330e-02 7.53934011e-02 5.60265258e-...
4,4,GCTTGGGTATATATTGTCCAATATAGCAGGCCTCATGTGCTCCTTA...,0,[-7.35620633e-02 1.12132251e-01 7.15436339e-...


In [9]:
test_df.shape

(400, 4)

In [10]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, Dataset
import faiss
from sklearn.metrics import accuracy_score, classification_report
from scipy.stats import mode
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("zhihan1996/DNABERT-2-117M", trust_remote_code=True)

# Load model using EsmModel
model =  AutoModel.from_pretrained("zhihan1996/DNABERT-2-117M", trust_remote_code=True,use_safetensors=True).to(device)
model.eval()

# Dataset + Dataloader
class SeqDataset(Dataset):
    def __init__(self, sequences):
        self.sequences = sequences
    def __len__(self):
        return len(self.sequences)
    def __getitem__(self, idx):
        return self.sequences[idx]

def collate_fn(batch):
    return tokenizer(batch, return_tensors='pt', padding=True, truncation=True)

def generate_embeddings(sequences, batch_size=32):
    dataset = SeqDataset(sequences)
    loader = DataLoader(dataset, batch_size=batch_size, collate_fn=collate_fn)
    all_embeddings = []

    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            pooled = outputs[0].mean(dim=1)  # Mean pooling
            all_embeddings.append(pooled.cpu())

    return torch.cat(all_embeddings).numpy()

# Start timer
start_time = time.time()

# Generate embeddings
train_embeddings = generate_embeddings(train_df["sequence"].tolist())
test_embeddings = generate_embeddings(test_df["sequence"].tolist())

# FAISS index (CPU)
dim = train_embeddings.shape[1]
index = faiss.IndexFlatL2(dim)  # Euclidean distance
index.add(train_embeddings)

# kNN search
k = 5
_, indices = index.search(test_embeddings, k)

# Retrieve training labels and majority vote
retrieved_labels = train_df.iloc[indices.flatten()]["target"].values.reshape(-1, k)
predicted_labels, _ = mode(retrieved_labels, axis=1)
predicted_labels = predicted_labels.squeeze()

# Evaluation
true_labels = test_df["target"].values
print(f"Accuracy: {accuracy_score(true_labels, predicted_labels)}")
print(classification_report(true_labels, predicted_labels))

# Time reporting
elapsed = time.time() - start_time
print(f"\n🕒 Evaluation took {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")


Some weights of BertModel were not initialized from the model checkpoint at zhihan1996/DNABERT-2-117M and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
C:\Users\FA007\.cache\huggingface\modules\transformers_modules\zhihan1996\DNABERT-2-117M\7bce263b15377fc15361f52cfab88f8b586abda0\bert_layers.py:433: UserWarning: Increasing alibi size from 512 to 549
  warnings.warn(


Accuracy: 0.65
              precision    recall  f1-score   support

           0       0.65      0.66      0.65       200
           1       0.65      0.64      0.65       200

    accuracy                           0.65       400
   macro avg       0.65      0.65      0.65       400
weighted avg       0.65      0.65      0.65       400


🕒 Evaluation took 525.74 seconds (8.76 minutes)


In [11]:
import triton
print(triton.__version__)


3.4.0


In [14]:

import numpy as np
import faiss
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report

# =============================================================================
# 1. Define Feature Extraction Functions
# =============================================================================

def compute_gc_content(seq):
    """
    Compute GC content (% of G and C bases) for a DNA/RNA sequence.
    Returns a fraction (between 0 and 1).
    """
    seq = seq.upper()
    count = sum(seq.count(base) for base in ['A', 'C', 'G', 'T'])
    if count == 0:
        return 0
    return (seq.count('G') + seq.count('C')) / count

def compute_zcurve(seq):
    """
    Compute the zCurve features:
      x = (A+G) - (C+T)
      y = (A+C) - (G+T)
      z = (A+T) - (G+C)
    Returns a numpy array with 3 values.
    """
    seq = seq.upper()
    A = seq.count('A')
    C = seq.count('C')
    G = seq.count('G')
    T = seq.count('T')
    x = (A + G) - (C + T)
    y = (A + C) - (G + T)
    z = (A + T) - (G + C)
    return np.array([x, y, z], dtype=float)

def compute_atgc_ratio(seq):
    """
    Compute the ratio (A+T) / (G+C).
    Returns a single numeric value (with safeguard for division by zero).
    """
    seq = seq.upper()
    A = seq.count('A')
    T = seq.count('T')
    G = seq.count('G')
    C = seq.count('C')
    denom = G + C
    if denom == 0:
        return 0
    return (A + T) / denom

def compute_cumulative_skew(seq):
    """
    Compute two skew metrics:
      - gcSkew = (G - C)/(G+C)
      - atSkew = (A - T)/(A+T)
    Returns a numpy array with 2 values.
    """
    seq = seq.upper()
    A = seq.count('A')
    T = seq.count('T')
    G = seq.count('G')
    C = seq.count('C')
    gcSkew = (G - C) / (G + C) if (G + C) != 0 else 0
    atSkew = (A - T) / (A + T) if (A + T) != 0 else 0
    return np.array([gcSkew, atSkew], dtype=float)

def compute_pseudoKNC(seq, k=3):
    """
    A placeholder implementation for pseudoKNC:
      Count frequencies for all k-mers.
    For DNA with k=3, this creates a feature vector for all 4^3 = 64 k-mers.
    (The exact implementation for pseudoKNC may include additional biochemical properties.)
    """
    from itertools import product
    seq = seq.upper()
    kmer_list = [''.join(p) for p in product('ACGT', repeat=k)]
    counts = np.array([seq.count(kmer) for kmer in kmer_list], dtype=float)
    if counts.sum() != 0:
        counts /= counts.sum()  # Normalize to frequencies
    return counts



# =============================================================================
# 2. Create a Modular Feature Extraction Pipeline
# =============================================================================

def extract_features(seq, feature_functions):
    """
    Given a sequence and a list of functions,
    compute each feature and concatenate them into one feature vector.
    """
    features = []
    for func in feature_functions:
        feat = func(seq)
        # Ensure scalar outputs become 1D arrays
        if np.isscalar(feat):
            feat = np.array([feat])
        features.append(feat)
    return np.concatenate(features)
import time

# Start timer
start_time = time.time()
# Create a dictionary mapping names to feature extraction functions.
feature_functions = {
    'gcContent': compute_gc_content,         # 1 feature
    'zCurve': compute_zcurve,                  # 3 features
    'atgcRatio': compute_atgc_ratio,           # 1 feature
    'cumulativeSkew': compute_cumulative_skew, # 2 features
    'pseudoKNC': lambda seq: compute_pseudoKNC(seq, k=3)  # 64 features (example)
    # Add other functions as needed.
}

# Choose the features you want to try. For example:
# To try a combination of gcContent + zCurve:
selected_feature_keys = [ 'gcContent']
selected_functions = [feature_functions[key] for key in selected_feature_keys]

# =============================================================================
# 3. Compute and Normalize Extracted Features for Train and Test Data
# =============================================================================

# Assume train_df and test_df are already defined and contain a 'sequence' column.
train_features = train_df['sequence'].apply(lambda seq: extract_features(seq, selected_functions))
test_features = test_df['sequence'].apply(lambda seq: extract_features(seq, selected_functions))

# Convert from Series of arrays to a 2D numpy array
train_features = np.stack(train_features.values)
test_features = np.stack(test_features.values)

# Normalize the extracted features using MinMaxScaler
scaler_feat = MinMaxScaler()
train_features_scaled = scaler_feat.fit_transform(train_features)
test_features_scaled = scaler_feat.transform(test_features)

# =============================================================================
# 4. Combine with Precomputed Embeddings
# =============================================================================

def normalize_embeddings(embeddings):
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    return embeddings / norms

# Assume embeddings are stored in the 'embedding' column as arrays.
train_embeddings = normalize_embeddings(train_embeddings)
test_embeddings = normalize_embeddings(test_embeddings)

# Set weights for the combination; adjust as needed.
alpha_emb = 0.8# Weight for the original embeddings
beta_feat = 0.2 # Weight for the extracted features

train_combined = np.hstack([
    alpha_emb * train_embeddings,
    beta_feat * train_features_scaled
])
test_combined = np.hstack([
    alpha_emb * test_embeddings,
    beta_feat * test_features_scaled
])

# =============================================================================
# 5. Build FAISS Index and Define Retrieval + Voting Functions
# =============================================================================

# Build the FAISS index (using inner product, which is equivalent to cosine similarity on normalized vectors)
index = faiss.IndexFlatIP(train_combined.shape[1])
index.add(train_combined)



def batch_dynamic_retrieval(test_comb, base_k=5, sim_thr=0.7):
    """
    1) Retrieve top max_k neighbors for all queries in one FAISS call.
    2) Return distances, indices, and a boolean mask (use_expanded).
    """
    n = test_comb.shape[0]
    max_k = min(base_k * 2, train_combined.shape[0])

    # 1) Single batched FAISS search → shape (n, max_k)
    distances, indices = index.search(test_combined, max_k)

    # 2) Decide per-query whether to expand
    use_expanded = distances[:, :base_k].mean(axis=1) < sim_thr

    return distances, indices, use_expanded

# Run batch retrieval once
D_all, I_all, expand_mask = batch_dynamic_retrieval(test_combined, base_k=5, sim_thr=0.7)

# 3) Weighted voting with dynamic k in the loop
preds = []
for i, (D_full, I_full) in enumerate(zip(D_all, I_all)):
    if expand_mask[i]:
        D_i, I_i = D_full, I_full
    else:
        D_i, I_i = D_full[:5], I_full[:5]  # base_k=5
    weights = 1 / (1 + np.exp(-D_i))
    class_w = np.zeros(len(train_df['target'].unique()), dtype=float)
    for w, idx in zip(weights, I_i):
        class_w[ train_df.iloc[idx]['target'] ] += w
    preds.append(class_w.argmax())

# 4) Final evaluation
print(classification_report(test_df['target'], preds))
end_time = time.time()
elapsed = end_time - start_time

# Report time
print(f"\n🕒 Evaluation took {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")


              precision    recall  f1-score   support

           0       0.67      0.58      0.62       200
           1       0.63      0.72      0.67       200

    accuracy                           0.65       400
   macro avg       0.65      0.65      0.65       400
weighted avg       0.65      0.65      0.65       400


🕒 Evaluation took 2.61 seconds (0.04 minutes)


In [15]:

import numpy as np
import faiss
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report

# =============================================================================
# 1. Define Feature Extraction Functions
# =============================================================================

def compute_gc_content(seq):
    """
    Compute GC content (% of G and C bases) for a DNA/RNA sequence.
    Returns a fraction (between 0 and 1).
    """
    seq = seq.upper()
    count = sum(seq.count(base) for base in ['A', 'C', 'G', 'T'])
    if count == 0:
        return 0
    return (seq.count('G') + seq.count('C')) / count

def compute_zcurve(seq):
    """
    Compute the zCurve features:
      x = (A+G) - (C+T)
      y = (A+C) - (G+T)
      z = (A+T) - (G+C)
    Returns a numpy array with 3 values.
    """
    seq = seq.upper()
    A = seq.count('A')
    C = seq.count('C')
    G = seq.count('G')
    T = seq.count('T')
    x = (A + G) - (C + T)
    y = (A + C) - (G + T)
    z = (A + T) - (G + C)
    return np.array([x, y, z], dtype=float)

def compute_atgc_ratio(seq):
    """
    Compute the ratio (A+T) / (G+C).
    Returns a single numeric value (with safeguard for division by zero).
    """
    seq = seq.upper()
    A = seq.count('A')
    T = seq.count('T')
    G = seq.count('G')
    C = seq.count('C')
    denom = G + C
    if denom == 0:
        return 0
    return (A + T) / denom

def compute_cumulative_skew(seq):
    """
    Compute two skew metrics:
      - gcSkew = (G - C)/(G+C)
      - atSkew = (A - T)/(A+T)
    Returns a numpy array with 2 values.
    """
    seq = seq.upper()
    A = seq.count('A')
    T = seq.count('T')
    G = seq.count('G')
    C = seq.count('C')
    gcSkew = (G - C) / (G + C) if (G + C) != 0 else 0
    atSkew = (A - T) / (A + T) if (A + T) != 0 else 0
    return np.array([gcSkew, atSkew], dtype=float)

def compute_pseudoKNC(seq, k=3):
    """
    A placeholder implementation for pseudoKNC:
      Count frequencies for all k-mers.
    For DNA with k=3, this creates a feature vector for all 4^3 = 64 k-mers.
    (The exact implementation for pseudoKNC may include additional biochemical properties.)
    """
    from itertools import product
    seq = seq.upper()
    kmer_list = [''.join(p) for p in product('ACGT', repeat=k)]
    counts = np.array([seq.count(kmer) for kmer in kmer_list], dtype=float)
    if counts.sum() != 0:
        counts /= counts.sum()  # Normalize to frequencies
    return counts



# =============================================================================
# 2. Create a Modular Feature Extraction Pipeline
# =============================================================================

def extract_features(seq, feature_functions):
    """
    Given a sequence and a list of functions,
    compute each feature and concatenate them into one feature vector.
    """
    features = []
    for func in feature_functions:
        feat = func(seq)
        # Ensure scalar outputs become 1D arrays
        if np.isscalar(feat):
            feat = np.array([feat])
        features.append(feat)
    return np.concatenate(features)
import time

# Start timer
start_time = time.time()
# Create a dictionary mapping names to feature extraction functions.
feature_functions = {
    'gcContent': compute_gc_content,         # 1 feature
    'zCurve': compute_zcurve,                  # 3 features
    'atgcRatio': compute_atgc_ratio,           # 1 feature
    'cumulativeSkew': compute_cumulative_skew, # 2 features
    'pseudoKNC': lambda seq: compute_pseudoKNC(seq, k=3)  # 64 features (example)

}

# Choose the features you want to try.

selected_feature_keys = [ 'zCurve']
selected_functions = [feature_functions[key] for key in selected_feature_keys]

# =============================================================================
# 3. Compute and Normalize Extracted Features for Train and Test Data
# =============================================================================

# Assume train_df and test_df are already defined and contain a 'sequence' column.
train_features = train_df['sequence'].apply(lambda seq: extract_features(seq, selected_functions))
test_features = test_df['sequence'].apply(lambda seq: extract_features(seq, selected_functions))

# Convert from Series of arrays to a 2D numpy array
train_features = np.stack(train_features.values)
test_features = np.stack(test_features.values)

# Normalize the extracted features using MinMaxScaler
scaler_feat = MinMaxScaler()
train_features_scaled = scaler_feat.fit_transform(train_features)
test_features_scaled = scaler_feat.transform(test_features)

# =============================================================================
# 4. Combine with Precomputed Embeddings
# =============================================================================

def normalize_embeddings(embeddings):
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    return embeddings / norms

# Assume embeddings are stored in the 'embedding' column as arrays.
train_embeddings = normalize_embeddings(train_embeddings)
test_embeddings = normalize_embeddings(test_embeddings)

# Set weights for the combination; adjust as needed.
alpha_emb = 0.8# Weight for the original embeddings
beta_feat = 0.2 # Weight for the extracted features

train_combined = np.hstack([
    alpha_emb * train_embeddings,
    beta_feat * train_features_scaled
])
test_combined = np.hstack([
    alpha_emb * test_embeddings,
    beta_feat * test_features_scaled
])

# =============================================================================
# 5. Build FAISS Index and Define Retrieval + Voting Functions
# =============================================================================

# Build the FAISS index (using inner product, which is equivalent to cosine similarity on normalized vectors)
index = faiss.IndexFlatIP(train_combined.shape[1])
index.add(train_combined)



def batch_dynamic_retrieval(test_comb, base_k=5, sim_thr=0.7):
    """
    1) Retrieve top max_k neighbors for all queries in one FAISS call.
    2) Return distances, indices, and a boolean mask (use_expanded).
    """
    n = test_comb.shape[0]
    max_k = min(base_k * 2, train_combined.shape[0])

    # 1) Single batched FAISS search → shape (n, max_k)
    distances, indices = index.search(test_combined, max_k)

    # 2) Decide per-query whether to expand
    use_expanded = distances[:, :base_k].mean(axis=1) < sim_thr

    return distances, indices, use_expanded

# Run batch retrieval once
D_all, I_all, expand_mask = batch_dynamic_retrieval(test_combined, base_k=5, sim_thr=0.7)

# 3) Weighted voting with dynamic k in the loop
preds = []
for i, (D_full, I_full) in enumerate(zip(D_all, I_all)):
    if expand_mask[i]:
        D_i, I_i = D_full, I_full
    else:
        D_i, I_i = D_full[:5], I_full[:5]  # base_k=5
    weights = 1 / (1 + np.exp(-D_i))
    class_w = np.zeros(len(train_df['target'].unique()), dtype=float)
    for w, idx in zip(weights, I_i):
        class_w[ train_df.iloc[idx]['target'] ] += w
    preds.append(class_w.argmax())

# 4) Final evaluation
print(classification_report(test_df['target'], preds))
end_time = time.time()
elapsed = end_time - start_time

# Report time
print(f"\n🕒 Evaluation took {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")


              precision    recall  f1-score   support

           0       0.61      0.82      0.70       200
           1       0.73      0.47      0.57       200

    accuracy                           0.65       400
   macro avg       0.67      0.65      0.63       400
weighted avg       0.67      0.65      0.63       400


🕒 Evaluation took 2.49 seconds (0.04 minutes)


In [17]:

import numpy as np
import faiss
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report

# =============================================================================
# 1. Define Feature Extraction Functions
# =============================================================================

def compute_gc_content(seq):
    """
    Compute GC content (% of G and C bases) for a DNA/RNA sequence.
    Returns a fraction (between 0 and 1).
    """
    seq = seq.upper()
    count = sum(seq.count(base) for base in ['A', 'C', 'G', 'T'])
    if count == 0:
        return 0
    return (seq.count('G') + seq.count('C')) / count

def compute_zcurve(seq):
    """
    Compute the zCurve features:
      x = (A+G) - (C+T)
      y = (A+C) - (G+T)
      z = (A+T) - (G+C)
    Returns a numpy array with 3 values.
    """
    seq = seq.upper()
    A = seq.count('A')
    C = seq.count('C')
    G = seq.count('G')
    T = seq.count('T')
    x = (A + G) - (C + T)
    y = (A + C) - (G + T)
    z = (A + T) - (G + C)
    return np.array([x, y, z], dtype=float)

def compute_atgc_ratio(seq):
    """
    Compute the ratio (A+T) / (G+C).
    Returns a single numeric value (with safeguard for division by zero).
    """
    seq = seq.upper()
    A = seq.count('A')
    T = seq.count('T')
    G = seq.count('G')
    C = seq.count('C')
    denom = G + C
    if denom == 0:
        return 0
    return (A + T) / denom

def compute_cumulative_skew(seq):
    """
    Compute two skew metrics:
      - gcSkew = (G - C)/(G+C)
      - atSkew = (A - T)/(A+T)
    Returns a numpy array with 2 values.
    """
    seq = seq.upper()
    A = seq.count('A')
    T = seq.count('T')
    G = seq.count('G')
    C = seq.count('C')
    gcSkew = (G - C) / (G + C) if (G + C) != 0 else 0
    atSkew = (A - T) / (A + T) if (A + T) != 0 else 0
    return np.array([gcSkew, atSkew], dtype=float)

def compute_pseudoKNC(seq, k=3):
    """
    A placeholder implementation for pseudoKNC:
      Count frequencies for all k-mers.
    For DNA with k=3, this creates a feature vector for all 4^3 = 64 k-mers.
    (The exact implementation for pseudoKNC may include additional biochemical properties.)
    """
    from itertools import product
    seq = seq.upper()
    kmer_list = [''.join(p) for p in product('ACGT', repeat=k)]
    counts = np.array([seq.count(kmer) for kmer in kmer_list], dtype=float)
    if counts.sum() != 0:
        counts /= counts.sum()  # Normalize to frequencies
    return counts



# =============================================================================
# 2. Create a Modular Feature Extraction Pipeline
# =============================================================================

def extract_features(seq, feature_functions):
    """
    Given a sequence and a list of functions,
    compute each feature and concatenate them into one feature vector.
    """
    features = []
    for func in feature_functions:
        feat = func(seq)
        # Ensure scalar outputs become 1D arrays
        if np.isscalar(feat):
            feat = np.array([feat])
        features.append(feat)
    return np.concatenate(features)
import time

# Start timer
start_time = time.time()
# Create a dictionary mapping names to feature extraction functions.
feature_functions = {
    'gcContent': compute_gc_content,         # 1 feature
    'zCurve': compute_zcurve,                  # 3 features
    'atgcRatio': compute_atgc_ratio,           # 1 feature
    'cumulativeSkew': compute_cumulative_skew, # 2 features
    'pseudoKNC': lambda seq: compute_pseudoKNC(seq, k=3)  # 64 features (example)

}

# Choose the features you want to try.

selected_feature_keys = ['atgcRatio']
selected_functions = [feature_functions[key] for key in selected_feature_keys]

# =============================================================================
# 3. Compute and Normalize Extracted Features for Train and Test Data
# =============================================================================

# Assume train_df and test_df are already defined and contain a 'sequence' column.
train_features = train_df['sequence'].apply(lambda seq: extract_features(seq, selected_functions))
test_features = test_df['sequence'].apply(lambda seq: extract_features(seq, selected_functions))

# Convert from Series of arrays to a 2D numpy array
train_features = np.stack(train_features.values)
test_features = np.stack(test_features.values)

# Normalize the extracted features using MinMaxScaler
scaler_feat = MinMaxScaler()
train_features_scaled = scaler_feat.fit_transform(train_features)
test_features_scaled = scaler_feat.transform(test_features)

# =============================================================================
# 4. Combine with Precomputed Embeddings
# =============================================================================

def normalize_embeddings(embeddings):
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    return embeddings / norms

# Assume embeddings are stored in the 'embedding' column as arrays.
train_embeddings = normalize_embeddings(train_embeddings)
test_embeddings = normalize_embeddings(test_embeddings)

# Set weights for the combination; adjust as needed.
alpha_emb = 0.8# Weight for the original embeddings
beta_feat = 0.2 # Weight for the extracted features

train_combined = np.hstack([
    alpha_emb * train_embeddings,
    beta_feat * train_features_scaled
])
test_combined = np.hstack([
    alpha_emb * test_embeddings,
    beta_feat * test_features_scaled
])

# =============================================================================
# 5. Build FAISS Index and Define Retrieval + Voting Functions
# =============================================================================

# Build the FAISS index (using inner product, which is equivalent to cosine similarity on normalized vectors)
index = faiss.IndexFlatIP(train_combined.shape[1])
index.add(train_combined)



def batch_dynamic_retrieval(test_comb, base_k=5, sim_thr=0.7):
    """
    1) Retrieve top max_k neighbors for all queries in one FAISS call.
    2) Return distances, indices, and a boolean mask (use_expanded).
    """
    n = test_comb.shape[0]
    max_k = min(base_k * 2, train_combined.shape[0])

    # 1) Single batched FAISS search → shape (n, max_k)
    distances, indices = index.search(test_combined, max_k)

    # 2) Decide per-query whether to expand
    use_expanded = distances[:, :base_k].mean(axis=1) < sim_thr

    return distances, indices, use_expanded

# Run batch retrieval once
D_all, I_all, expand_mask = batch_dynamic_retrieval(test_combined, base_k=5, sim_thr=0.7)

# 3) Weighted voting with dynamic k in the loop
preds = []
for i, (D_full, I_full) in enumerate(zip(D_all, I_all)):
    if expand_mask[i]:
        D_i, I_i = D_full, I_full
    else:
        D_i, I_i = D_full[:5], I_full[:5]  # base_k=5
    weights = 1 / (1 + np.exp(-D_i))
    class_w = np.zeros(len(train_df['target'].unique()), dtype=float)
    for w, idx in zip(weights, I_i):
        class_w[ train_df.iloc[idx]['target'] ] += w
    preds.append(class_w.argmax())

# 4) Final evaluation
print(classification_report(test_df['target'], preds))
end_time = time.time()
elapsed = end_time - start_time

# Report time
print(f"\n🕒 Evaluation took {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")


              precision    recall  f1-score   support

           0       0.67      0.66      0.66       200
           1       0.67      0.68      0.67       200

    accuracy                           0.67       400
   macro avg       0.67      0.67      0.67       400
weighted avg       0.67      0.67      0.67       400


🕒 Evaluation took 2.24 seconds (0.04 minutes)


In [19]:

import numpy as np
import faiss
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report

# =============================================================================
# 1. Define Feature Extraction Functions
# =============================================================================

def compute_gc_content(seq):
    """
    Compute GC content (% of G and C bases) for a DNA/RNA sequence.
    Returns a fraction (between 0 and 1).
    """
    seq = seq.upper()
    count = sum(seq.count(base) for base in ['A', 'C', 'G', 'T'])
    if count == 0:
        return 0
    return (seq.count('G') + seq.count('C')) / count

def compute_zcurve(seq):
    """
    Compute the zCurve features:
      x = (A+G) - (C+T)
      y = (A+C) - (G+T)
      z = (A+T) - (G+C)
    Returns a numpy array with 3 values.
    """
    seq = seq.upper()
    A = seq.count('A')
    C = seq.count('C')
    G = seq.count('G')
    T = seq.count('T')
    x = (A + G) - (C + T)
    y = (A + C) - (G + T)
    z = (A + T) - (G + C)
    return np.array([x, y, z], dtype=float)

def compute_atgc_ratio(seq):
    """
    Compute the ratio (A+T) / (G+C).
    Returns a single numeric value (with safeguard for division by zero).
    """
    seq = seq.upper()
    A = seq.count('A')
    T = seq.count('T')
    G = seq.count('G')
    C = seq.count('C')
    denom = G + C
    if denom == 0:
        return 0
    return (A + T) / denom

def compute_cumulative_skew(seq):
    """
    Compute two skew metrics:
      - gcSkew = (G - C)/(G+C)
      - atSkew = (A - T)/(A+T)
    Returns a numpy array with 2 values.
    """
    seq = seq.upper()
    A = seq.count('A')
    T = seq.count('T')
    G = seq.count('G')
    C = seq.count('C')
    gcSkew = (G - C) / (G + C) if (G + C) != 0 else 0
    atSkew = (A - T) / (A + T) if (A + T) != 0 else 0
    return np.array([gcSkew, atSkew], dtype=float)

def compute_pseudoKNC(seq, k=3):
    """
    A placeholder implementation for pseudoKNC:
      Count frequencies for all k-mers.
    For DNA with k=3, this creates a feature vector for all 4^3 = 64 k-mers.
    (The exact implementation for pseudoKNC may include additional biochemical properties.)
    """
    from itertools import product
    seq = seq.upper()
    kmer_list = [''.join(p) for p in product('ACGT', repeat=k)]
    counts = np.array([seq.count(kmer) for kmer in kmer_list], dtype=float)
    if counts.sum() != 0:
        counts /= counts.sum()  # Normalize to frequencies
    return counts



# =============================================================================
# 2. Create a Modular Feature Extraction Pipeline
# =============================================================================

def extract_features(seq, feature_functions):
    """
    Given a sequence and a list of functions,
    compute each feature and concatenate them into one feature vector.
    """
    features = []
    for func in feature_functions:
        feat = func(seq)
        # Ensure scalar outputs become 1D arrays
        if np.isscalar(feat):
            feat = np.array([feat])
        features.append(feat)
    return np.concatenate(features)
import time

# Start timer
start_time = time.time()
# Create a dictionary mapping names to feature extraction functions.
feature_functions = {
    'gcContent': compute_gc_content,         # 1 feature
    'zCurve': compute_zcurve,                  # 3 features
    'atgcRatio': compute_atgc_ratio,           # 1 feature
    'cumulativeSkew': compute_cumulative_skew, # 2 features
    'pseudoKNC': lambda seq: compute_pseudoKNC(seq, k=3)  # 64 features (example)

}

# Choose the features you want to try.

selected_feature_keys = ['cumulativeSkew']
selected_functions = [feature_functions[key] for key in selected_feature_keys]

# =============================================================================
# 3. Compute and Normalize Extracted Features for Train and Test Data
# =============================================================================

# Assume train_df and test_df are already defined and contain a 'sequence' column.
train_features = train_df['sequence'].apply(lambda seq: extract_features(seq, selected_functions))
test_features = test_df['sequence'].apply(lambda seq: extract_features(seq, selected_functions))

# Convert from Series of arrays to a 2D numpy array
train_features = np.stack(train_features.values)
test_features = np.stack(test_features.values)

# Normalize the extracted features using MinMaxScaler
scaler_feat = MinMaxScaler()
train_features_scaled = scaler_feat.fit_transform(train_features)
test_features_scaled = scaler_feat.transform(test_features)

# =============================================================================
# 4. Combine with Precomputed Embeddings
# =============================================================================

def normalize_embeddings(embeddings):
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    return embeddings / norms

# Assume embeddings are stored in the 'embedding' column as arrays.
train_embeddings = normalize_embeddings(train_embeddings)
test_embeddings = normalize_embeddings(test_embeddings)

# Set weights for the combination; adjust as needed.
alpha_emb = 0.8# Weight for the original embeddings
beta_feat = 0.2 # Weight for the extracted features

train_combined = np.hstack([
    alpha_emb * train_embeddings,
    beta_feat * train_features_scaled
])
test_combined = np.hstack([
    alpha_emb * test_embeddings,
    beta_feat * test_features_scaled
])

# =============================================================================
# 5. Build FAISS Index and Define Retrieval + Voting Functions
# =============================================================================

# Build the FAISS index (using inner product, which is equivalent to cosine similarity on normalized vectors)
index = faiss.IndexFlatIP(train_combined.shape[1])
index.add(train_combined)



def batch_dynamic_retrieval(test_comb, base_k=5, sim_thr=0.7):
    """
    1) Retrieve top max_k neighbors for all queries in one FAISS call.
    2) Return distances, indices, and a boolean mask (use_expanded).
    """
    n = test_comb.shape[0]
    max_k = min(base_k * 2, train_combined.shape[0])

    # 1) Single batched FAISS search → shape (n, max_k)
    distances, indices = index.search(test_combined, max_k)

    # 2) Decide per-query whether to expand
    use_expanded = distances[:, :base_k].mean(axis=1) < sim_thr

    return distances, indices, use_expanded

# Run batch retrieval once
D_all, I_all, expand_mask = batch_dynamic_retrieval(test_combined, base_k=5, sim_thr=0.7)

# 3) Weighted voting with dynamic k in the loop
preds = []
for i, (D_full, I_full) in enumerate(zip(D_all, I_all)):
    if expand_mask[i]:
        D_i, I_i = D_full, I_full
    else:
        D_i, I_i = D_full[:5], I_full[:5]  # base_k=5
    weights = 1 / (1 + np.exp(-D_i))
    class_w = np.zeros(len(train_df['target'].unique()), dtype=float)
    for w, idx in zip(weights, I_i):
        class_w[ train_df.iloc[idx]['target'] ] += w
    preds.append(class_w.argmax())

# 4) Final evaluation
print(classification_report(test_df['target'], preds))
end_time = time.time()
elapsed = end_time - start_time

# Report time
print(f"\n🕒 Evaluation took {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")


              precision    recall  f1-score   support

           0       0.67      0.68      0.67       200
           1       0.67      0.66      0.67       200

    accuracy                           0.67       400
   macro avg       0.67      0.67      0.67       400
weighted avg       0.67      0.67      0.67       400


🕒 Evaluation took 2.29 seconds (0.04 minutes)


In [20]:

import numpy as np
import faiss
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report

# =============================================================================
# 1. Define Feature Extraction Functions
# =============================================================================

def compute_gc_content(seq):
    """
    Compute GC content (% of G and C bases) for a DNA/RNA sequence.
    Returns a fraction (between 0 and 1).
    """
    seq = seq.upper()
    count = sum(seq.count(base) for base in ['A', 'C', 'G', 'T'])
    if count == 0:
        return 0
    return (seq.count('G') + seq.count('C')) / count

def compute_zcurve(seq):
    """
    Compute the zCurve features:
      x = (A+G) - (C+T)
      y = (A+C) - (G+T)
      z = (A+T) - (G+C)
    Returns a numpy array with 3 values.
    """
    seq = seq.upper()
    A = seq.count('A')
    C = seq.count('C')
    G = seq.count('G')
    T = seq.count('T')
    x = (A + G) - (C + T)
    y = (A + C) - (G + T)
    z = (A + T) - (G + C)
    return np.array([x, y, z], dtype=float)

def compute_atgc_ratio(seq):
    """
    Compute the ratio (A+T) / (G+C).
    Returns a single numeric value (with safeguard for division by zero).
    """
    seq = seq.upper()
    A = seq.count('A')
    T = seq.count('T')
    G = seq.count('G')
    C = seq.count('C')
    denom = G + C
    if denom == 0:
        return 0
    return (A + T) / denom

def compute_cumulative_skew(seq):
    """
    Compute two skew metrics:
      - gcSkew = (G - C)/(G+C)
      - atSkew = (A - T)/(A+T)
    Returns a numpy array with 2 values.
    """
    seq = seq.upper()
    A = seq.count('A')
    T = seq.count('T')
    G = seq.count('G')
    C = seq.count('C')
    gcSkew = (G - C) / (G + C) if (G + C) != 0 else 0
    atSkew = (A - T) / (A + T) if (A + T) != 0 else 0
    return np.array([gcSkew, atSkew], dtype=float)

def compute_pseudoKNC(seq, k=3):
    """
    A placeholder implementation for pseudoKNC:
      Count frequencies for all k-mers.
    For DNA with k=3, this creates a feature vector for all 4^3 = 64 k-mers.
    (The exact implementation for pseudoKNC may include additional biochemical properties.)
    """
    from itertools import product
    seq = seq.upper()
    kmer_list = [''.join(p) for p in product('ACGT', repeat=k)]
    counts = np.array([seq.count(kmer) for kmer in kmer_list], dtype=float)
    if counts.sum() != 0:
        counts /= counts.sum()  # Normalize to frequencies
    return counts



# =============================================================================
# 2. Create a Modular Feature Extraction Pipeline
# =============================================================================

def extract_features(seq, feature_functions):
    """
    Given a sequence and a list of functions,
    compute each feature and concatenate them into one feature vector.
    """
    features = []
    for func in feature_functions:
        feat = func(seq)
        # Ensure scalar outputs become 1D arrays
        if np.isscalar(feat):
            feat = np.array([feat])
        features.append(feat)
    return np.concatenate(features)
import time

# Start timer
start_time = time.time()
# Create a dictionary mapping names to feature extraction functions.
feature_functions = {
    'gcContent': compute_gc_content,         # 1 feature
    'zCurve': compute_zcurve,                  # 3 features
    'atgcRatio': compute_atgc_ratio,           # 1 feature
    'cumulativeSkew': compute_cumulative_skew, # 2 features
    'pseudoKNC': lambda seq: compute_pseudoKNC(seq, k=3)  # 64 features (example)

}

# Choose the features you want to try.

selected_feature_keys = ['pseudoKNC']
selected_functions = [feature_functions[key] for key in selected_feature_keys]

# =============================================================================
# 3. Compute and Normalize Extracted Features for Train and Test Data
# =============================================================================

# Assume train_df and test_df are already defined and contain a 'sequence' column.
train_features = train_df['sequence'].apply(lambda seq: extract_features(seq, selected_functions))
test_features = test_df['sequence'].apply(lambda seq: extract_features(seq, selected_functions))

# Convert from Series of arrays to a 2D numpy array
train_features = np.stack(train_features.values)
test_features = np.stack(test_features.values)

# Normalize the extracted features using MinMaxScaler
scaler_feat = MinMaxScaler()
train_features_scaled = scaler_feat.fit_transform(train_features)
test_features_scaled = scaler_feat.transform(test_features)

# =============================================================================
# 4. Combine with Precomputed Embeddings
# =============================================================================

def normalize_embeddings(embeddings):
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    return embeddings / norms

# Assume embeddings are stored in the 'embedding' column as arrays.
train_embeddings = normalize_embeddings(train_embeddings)
test_embeddings = normalize_embeddings(test_embeddings)

# Set weights for the combination; adjust as needed.
alpha_emb = 0.8# Weight for the original embeddings
beta_feat = 0.2 # Weight for the extracted features

train_combined = np.hstack([
    alpha_emb * train_embeddings,
    beta_feat * train_features_scaled
])
test_combined = np.hstack([
    alpha_emb * test_embeddings,
    beta_feat * test_features_scaled
])

# =============================================================================
# 5. Build FAISS Index and Define Retrieval + Voting Functions
# =============================================================================

# Build the FAISS index (using inner product, which is equivalent to cosine similarity on normalized vectors)
index = faiss.IndexFlatIP(train_combined.shape[1])
index.add(train_combined)



def batch_dynamic_retrieval(test_comb, base_k=5, sim_thr=0.7):
    """
    1) Retrieve top max_k neighbors for all queries in one FAISS call.
    2) Return distances, indices, and a boolean mask (use_expanded).
    """
    n = test_comb.shape[0]
    max_k = min(base_k * 2, train_combined.shape[0])

    # 1) Single batched FAISS search → shape (n, max_k)
    distances, indices = index.search(test_combined, max_k)

    # 2) Decide per-query whether to expand
    use_expanded = distances[:, :base_k].mean(axis=1) < sim_thr

    return distances, indices, use_expanded

# Run batch retrieval once
D_all, I_all, expand_mask = batch_dynamic_retrieval(test_combined, base_k=5, sim_thr=0.7)

# 3) Weighted voting with dynamic k in the loop
preds = []
for i, (D_full, I_full) in enumerate(zip(D_all, I_all)):
    if expand_mask[i]:
        D_i, I_i = D_full, I_full
    else:
        D_i, I_i = D_full[:5], I_full[:5]  # base_k=5
    weights = 1 / (1 + np.exp(-D_i))
    class_w = np.zeros(len(train_df['target'].unique()), dtype=float)
    for w, idx in zip(weights, I_i):
        class_w[ train_df.iloc[idx]['target'] ] += w
    preds.append(class_w.argmax())

# 4) Final evaluation
print(classification_report(test_df['target'], preds))
end_time = time.time()
elapsed = end_time - start_time

# Report time
print(f"\n🕒 Evaluation took {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")


              precision    recall  f1-score   support

           0       0.67      0.67      0.67       200
           1       0.67      0.67      0.67       200

    accuracy                           0.67       400
   macro avg       0.67      0.67      0.67       400
weighted avg       0.67      0.67      0.67       400


🕒 Evaluation took 10.97 seconds (0.18 minutes)
